# Two argon atoms: a first MD run

| | |
|---|---|
| **Time** | About 30 minutes: four sections, each ending in a checkpoint. |
| **You will** | Run a two-atom simulation and read its energy trace. Predict, then measure, how the energy error grows with the time step. Explain, from the plan's ordering rules, why forces are computed before velocities are updated. Read the Lennard-Jones well depth off a run. |
| **Six ideas** | System, recipe, plan, the leap-frog step, time-step error, well depth. |
| **Not taught here** | Solvent, heat baths, constraints, trajectory analysis, how GROMOS topologies are built. |
| **Not for research** | newen is an educational reimplementation of GROMOS. It is not intended for research work; see the repository README. |

Every experiment starts with a **Predict** cell. Write your guess there before you run the
cell after it: a prediction made after seeing the answer teaches nothing.

## Setup

The input files come from the repository's reference suite: the same files the test suite
checks against gromosXX. The path assumes Jupyter was started in `py-newen/notebooks/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from newen import EnergyTimeseries, Plan, Recipe, Simulation, System
from newen.exceptions import PlanError

REF_DIR = Path("../../crates/newen-md/tests/gromosXX_references/pair_lj")

**If something fails.** The three most likely errors in this notebook, and their fixes:

| Error | Fix |
|---|---|
| `ModuleNotFoundError: No module named 'newen'` | The kernel has no py-newen. In `py-newen/`, run `uv sync --group notebooks`, then start Jupyter with `uv run --group notebooks jupyter lab`. |
| `OSError: Failed to read topology ... File not found` | Jupyter started in another folder. Set `REF_DIR` to `<your checkout>/crates/newen-md/tests/gromosXX_references/pair_lj`. |
| `RecipeError: Recipe.update: unknown field ...` | A misspelt recipe field. The message lists the valid names: the time step is `dt`, not `timestep`. |

## 1. The system and the recipe

A **system** is what is simulated. GROMOS describes it in two files. The *topology* says what
the atoms are and how they interact: masses, charges, Lennard-Jones parameters. The
*configuration* says where the atoms are, how fast they move, and what box they sit in.

Here the system is two argon atoms in vacuum, 0.35 nm apart and at rest.

In [ ]:
system = System.from_files(str(REF_DIR / "pair_lj.topo"), str(REF_DIR / "pair_lj.conf"))

print(system)
print("masses (u):", system.topology.masses)
print("positions (nm):")
print(system.positions)

Two argon atoms interact only through the Lennard-Jones potential. GROMOS writes it with two
parameters taken from the topology:

$$V(r) = \frac{C_{12}}{r^{12}} - \frac{C_6}{r^6}$$

For argon, $C_6 = 6.2647\times10^{-3}$ kJ mol⁻¹ nm⁶ and $C_{12} = 9.847\times10^{-6}$
kJ mol⁻¹ nm¹² (the `LJPARAMETERS` block of `pair_lj.topo`). The first term pushes the atoms
apart at short range; the second pulls them together at long range.

In [ ]:
def lj_energy(r: float | np.ndarray, c6: float, c12: float) -> float | np.ndarray:
    """Lennard-Jones energy in kJ/mol of one pair at distance r in nm."""
    return c12 / r**12 - c6 / r**6


C6, C12 = 6.2647e-3, 9.847e-6  # kJ mol^-1 nm^6, kJ mol^-1 nm^12

A **recipe** says how to run: the time step `dt`, the number of steps, the ensemble, the
cut-offs. This one is read from the reference input file `pair_lj.in`. It runs at constant
energy (NVE): there is no heat bath, so nothing adds or removes energy.

In [ ]:
recipe = Recipe.from_imd(str(REF_DIR / "pair_lj.in"))

print(recipe)
print("dt =", recipe.control["dt"], "ps,  steps =", recipe.control["steps"])

**Predict.** At step 0 the atoms are 0.35 nm apart. What is their potential energy? Is the
force between them attractive or repulsive? Use `lj_energy` or a calculator.

In [ ]:
my_energy_at_step_0 = None  # kJ/mol
my_force_is = None  # "attractive" or "repulsive"

Creating a `Simulation` already takes **step 0**: the engine evaluates the energy and the
forces at the starting positions, then moves the atoms once. So `sim.potential_energy` and
`sim.forces` below belong to the starting positions.

In [ ]:
sim = Simulation(system, recipe)

print(f"by hand: V(0.35 nm) = {lj_energy(0.35, C6, C12):.6f} kJ/mol")
print(f"engine:  V          = {sim.potential_energy:.6f} kJ/mol")
print(f"you predicted {my_energy_at_step_0} kJ/mol and a {my_force_is} force")
print("forces (kJ mol^-1 nm^-1):")
print(sim.forces)

assert np.isclose(sim.potential_energy, lj_energy(0.35, C6, C12), rtol=1e-9)

Atom 2 sits on the +x side and is pushed further along +x: the force is **repulsive**. The
atoms start closer than the distance of lowest energy, so they fly apart, slow down, and come
back.

A `Simulation` works on its own copy of the system, so `system` itself never moves. Every run
below starts from the same two atoms.

**Checkpoint 1.** You can load a system, read a recipe, and check the engine's step-0 energy
against the formula.

## 2. The plan: one MD step as data

The **plan** is the list of algorithms that make up one MD step, in the order they run.
`recipe.plan(system)` returns it. You can read it, edit it, and validate it.

In [ ]:
plan = recipe.plan(system)

for position, kind in enumerate(plan.kinds):
    print(position, kind)

| Step | What it does |
|---|---|
| `remove_com` | Removes centre-of-mass motion. Here there is none to remove. |
| `forcefield` | Forces $F(t)$ and potential energy at the current positions $x(t)$. |
| `leap_frog_velocity` | The **kick**: $v(t+\tfrac{\Delta t}{2}) = v(t-\tfrac{\Delta t}{2}) + \tfrac{F(t)}{m}\,\Delta t$ |
| `leap_frog_position` | The **drift**: $x(t+\Delta t) = x(t) + v(t+\tfrac{\Delta t}{2})\,\Delta t$ |
| `temperature_calculation` | Kinetic energy and temperature from the velocities. |
| `energy_calculation` | The totals: kinetic plus potential. |

Force, kick, drift is the **leap-frog step**. Velocities live at half steps and positions at
whole steps, and each leaps over the other.

**Predict.** The library refuses a plan that breaks the MD step. Which of these three edits
does it refuse?

- (a) drop `leap_frog_velocity`
- (b) move `forcefield` after `leap_frog_position`
- (c) put `leap_frog_position` before `leap_frog_velocity`

In [ ]:
my_refused = None  # a set, such as {"a", "b"}

In [ ]:
def verdict(plan: Plan) -> str:
    """'accepted', or the reason the library refuses the plan."""
    try:
        plan.validate()
    except PlanError as err:
        return f"refused: {err}"
    return "accepted"


plan_a = recipe.plan(system)
plan_a.remove("leap_frog_velocity")

plan_b = recipe.plan(system)
plan_b.insert_after("leap_frog_position", plan_b.remove("forcefield"))

plan_c = recipe.plan(system)
plan_c.insert_before("leap_frog_velocity", plan_c.remove("leap_frog_position"))

verdicts = {"a": verdict(plan_a), "b": verdict(plan_b), "c": verdict(plan_c)}
for name, text in verdicts.items():
    print(f"({name}) {text}")
print("you predicted these would be refused:", my_refused)

In [ ]:
assert verdicts == {
    "a": "refused: invalid algorithm plan: leap_frog_position requires leap_frog_velocity",
    "b": "refused: invalid algorithm plan: leap_frog_velocity must come after forcefield",
    "c": "refused: invalid algorithm plan: leap_frog_velocity must come before leap_frog_position",
}

All three are refused. The kick needs $F(t)$, the force at positions the drift has not moved
yet, so `forcefield` must come before `leap_frog_velocity` (b). The drift needs the new
half-step velocity, so the kick must come first (c) and cannot be left out (a). These are
GROMOS's own ordering rules, and `Simulation(system, recipe, plan=...)` checks them too: an
edited plan runs only if it passes.

**Checkpoint 2.** You can list the algorithms of one MD step and say why the force comes
before the kick.

## 3. The time step and the energy error

With no heat bath, the total energy should stay constant. Leap-frog keeps it constant only
approximately, and the deviation is the **time-step error**. First, look at one run: 8 ps at
the reference step of 0.002 ps.

In [ ]:
dt, n_steps = 0.002, 4000  # ps, steps: 8 ps in total
sim = Simulation(system, recipe.update(control={"dt": dt, "steps": n_steps}))
ts = EnergyTimeseries(sim.run(n_steps, ene_freq=1))

fig, ax = plt.subplots(figsize=(8, 3.5))
for name in ("kinetic", "potential", "total"):
    ax.plot(ts.time, getattr(ts, name), label=name)
ax.set_xlabel("time (ps)")
ax.set_ylabel("energy (kJ/mol)")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

print(f"total-energy range over 8 ps: {np.ptp(ts.total):.2e} kJ/mol")

The atoms swing apart and back once every 1.98 ps. The kinetic energy peaks twice per swing,
each time the atoms pass the bottom of the well. The total looks flat, but its range (largest minus smallest value) is not
zero. That range is the measure of the error used below.

**Predict.** Double the time step and keep the same 8 ps. By what factor does the range of
the total energy grow: 1, 2, 4 or 8?

In [ ]:
my_factor = None

In [ ]:
def energy_range(system: System, recipe: Recipe, dt: float, duration: float) -> float:
    """Largest minus smallest total energy (kJ/mol) over `duration` ps in steps of `dt` ps."""
    n_steps = round(duration / dt)
    sim = Simulation(system, recipe.update(control={"dt": dt, "steps": n_steps}))
    return float(np.ptp(EnergyTimeseries(sim.run(n_steps, ene_freq=1)).total))


dts = np.array([0.0005, 0.001, 0.002, 0.004, 0.008, 0.016, 0.032, 0.064, 0.128])
ranges = np.array([energy_range(system, recipe, dt, duration=8.0) for dt in dts])

for dt, energy_error in zip(dts, ranges, strict=True):
    print(f"dt = {dt:6.4f} ps   range = {energy_error:.3e} kJ/mol")

In [ ]:
small = dts <= 0.032
slope = np.polyfit(np.log(dts[small]), np.log(ranges[small]), 1)[0]
print(f"log-log slope up to 0.032 ps: {slope:.2f}")
print(f"doubling dt multiplies the error by 2**slope = {2**slope:.2f}; you predicted {my_factor}")

assert 1.9 <= slope <= 2.1

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dts, ranges, "o-", label="measured")
ax.loglog(dts, ranges[0] * (dts / dts[0]) ** 2, "--", label="slope 2")
ax.set_xlabel("time step dt (ps)")
ax.set_ylabel("total-energy range over 8 ps (kJ/mol)")
ax.legend()
plt.show()

The error grows as $\Delta t^2$: leap-frog is a **second-order** method. Halving the step cuts
the error by four, and costs twice as many steps.

The last points leave the line. At 0.128 ps one swing takes only about 15 steps, the step is
no longer small, and the error grows as large as the kinetic energy itself.

**Checkpoint 3.** You can measure the order of an integrator from a sweep of the time step.

## 4. The well depth

The **well depth** $\varepsilon$ is how far the potential dips below zero at its minimum. In
the $C_6$, $C_{12}$ form the minimum sits at $r_\mathrm{min} = (2C_{12}/C_6)^{1/6}$, where

$$V(r_\mathrm{min}) = -\varepsilon = -\frac{C_6^2}{4\,C_{12}}$$

In [ ]:
epsilon = C6**2 / (4 * C12)
r_min = (2 * C12 / C6) ** (1 / 6)
print(f"epsilon = {epsilon:.5f} kJ/mol at r_min = {r_min:.4f} nm")

**Predict.** The atoms start at 0.35 nm, at rest. Will the potential energy ever reach
$-\varepsilon$ during the run? If it does, where are the atoms at that moment?

In [ ]:
my_answer = None  # a sentence

In [ ]:
dt, n_steps = 0.0005, 16000  # ps, steps: 8 ps in total
sim = Simulation(system, recipe.update(control={"dt": dt, "steps": n_steps}))
ts = EnergyTimeseries(sim.run(n_steps, ene_freq=1))

lowest = ts.potential.min()
print(f"lowest potential energy in the run: {lowest:.5f} kJ/mol")
print(f"-epsilon from the parameters:       {-epsilon:.5f} kJ/mol")

assert abs(lowest / -epsilon - 1) < 1e-3

In [ ]:
r = np.linspace(0.33, 0.60, 300)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(r, lj_energy(r, C6, C12), label="V(r)")
ax.axhline(ts.total.mean(), ls="--", color="grey", label="total energy")
ax.plot(r_min, -epsilon, "o", label="minimum: -epsilon")
ax.plot(0.35, lj_energy(0.35, C6, C12), "s", label="start")
ax.set_ylim(-1.1, 0.5)
ax.set_xlabel("distance r (nm)")
ax.set_ylabel("energy (kJ/mol)")
ax.legend()
plt.show()

Yes. The total energy, −0.494 kJ/mol, lies above the bottom of the well, so the atoms swing
between two turning points where the dashed line meets the curve: 0.350 nm and 0.470 nm. They
pass $r_\mathrm{min}$ twice per swing. There the potential energy is $-\varepsilon$ and the
kinetic energy is at its largest.

**Checkpoint 4.** You can read $\varepsilon$ off a run and explain why the run reaches it.

## Summary

- A **system** is a topology plus a configuration; a **recipe** says how to run it.
- The **plan** is one MD step as ordered data: force, kick, drift. The library refuses an
  order that breaks it.
- Leap-frog's energy error grows as $\Delta t^2$.
- The Lennard-Jones well depth is $\varepsilon = C_6^2 / 4C_{12}$, and a bound pair passes
  through it on every swing.

Next: [`01_load_and_inspect.ipynb`](01_load_and_inspect.ipynb) loads larger systems, and
[`02_short_md.ipynb`](02_short_md.ipynb) adds solvent, heat baths and constraints.